# Tarea 9 · Revisión de calidad y dataset limpio final

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/main/notebooks/trabajo/9_dataset_limpio.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**
**Integrantes:** Matías, Julián y Emmanuel. **Responsable de esta tarea:** Emmanuel.

Tomamos `data/processed/telco_step2.csv` (tarea 7), aplicamos la decisión sobre valores atípicos de la tarea 8, revisamos la calidad de los datos y exportamos `data/processed/telco_clean.csv`, el dataset limpio que usan las tareas de EDA, visualización y modelado.

## Cómo ejecutarlo

- **Local:** instalá las dependencias de `requirements.txt`, abrí el notebook desde el repositorio y ejecutá todas las celdas.
- **Colab:** abrí el botón y elegí **Entorno de ejecución → Ejecutar todo**. La entrada se lee desde `main` de GitHub. El resultado queda en el panel **Archivos** y se puede descargar.

## 1. Carga de la salida de la tarea 7

Un CSV no guarda los tipos `category`, así que al leerlo indicamos el esquema acordado: 17 variables categóricas y 5 numéricas.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

GITHUB_REPO = 'julianschmidt1/tpi-analisis-de-datos'
GITHUB_REF = 'main'
RUTA_ENTRADA = Path('data/processed/telco_step2.csv')
URL_ENTRADA = (
    f'https://raw.githubusercontent.com/{GITHUB_REPO}/'
    f'{GITHUB_REF}/{RUTA_ENTRADA.as_posix()}'
)

if EN_COLAB:
    RAIZ = Path.cwd()
    fuente = URL_ENTRADA
else:
    actual = Path.cwd().resolve()
    RAIZ = next(
        (ruta for ruta in (actual, *actual.parents) if (ruta / RUTA_ENTRADA).is_file()),
        None,
    )
    if RAIZ is None:
        raise FileNotFoundError(
            f'No se encontró {RUTA_ENTRADA.as_posix()}. '
            'Actualizá el repositorio y abrí Jupyter dentro de él.'
        )
    fuente = RAIZ / RUTA_ENTRADA

columnas_categoricas = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
    'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn',
]
tipos_esperados = {
    **dict.fromkeys(columnas_categoricas, 'category'),
    'tenure': 'int64',
    'MonthlyCharges': 'float64',
    'TotalCharges': 'float64',
    'ingreso_estimado': 'float64',
    'diferencia_facturacion': 'float64',
}

df = pd.read_csv(fuente, dtype=tipos_esperados)

print(f'Entorno: {"Colab" if EN_COLAB else "local"}')
print(f'Fuente: {fuente}')
print(f'Entrada: {df.shape[0]:,} clientes y {df.shape[1]} columnas.')
df.info()

Entorno: local
Fuente: F:\codear\analisis-datos\tpi-analisis-de-datos\data\processed\telco_step2.csv
Entrada: 7,043 clientes y 22 columnas.
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   gender                  7043 non-null   category
 1   SeniorCitizen           7043 non-null   category
 2   Partner                 7043 non-null   category
 3   Dependents              7043 non-null   category
 4   tenure                  7043 non-null   int64   
 5   PhoneService            7043 non-null   category
 6   MultipleLines           7043 non-null   category
 7   InternetService         7043 non-null   category
 8   OnlineSecurity          7043 non-null   category
 9   OnlineBackup            7043 non-null   category
 10  DeviceProtection        7043 non-null   category
 11  TechSupport             7043 non-null   category
 12  Str

## 2. Decisión sobre valores atípicos (tarea 8)

Un valor atípico (*outlier*) es un dato muy alejado del resto. Se detecta con el rango intercuartílico (IQR): se marca como atípico lo que queda por debajo de Q1 − 1,5 × IQR o por encima de Q3 + 1,5 × IQR. En la tarea 8 se concluyó que **no hay atípicos** y que corresponde **conservar los 7.043 registros**. Repetimos el conteo para dejar la evidencia en este notebook.

In [2]:
variables_numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']
filas = []
for columna in variables_numericas:
    q1, q3 = df[columna].quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    n_atipicos = int(((df[columna] < limite_inferior) | (df[columna] > limite_superior)).sum())
    filas.append({
        'variable': columna,
        'limite_inferior': round(limite_inferior, 2),
        'limite_superior': round(limite_superior, 2),
        'atipicos': n_atipicos,
    })

resumen_atipicos = pd.DataFrame(filas).set_index('variable')
display(resumen_atipicos)
assert resumen_atipicos['atipicos'].sum() == 0, 'Hay atípicos: revisar la decisión de la tarea 8.'
print('Decisión aplicada: se conservan todos los registros (0 valores atípicos).')

,limite_inferior,limite_superior,atipicos
variable,,,
tenure,-60.00,124.00,0
MonthlyCharges,-46.02,171.38,0
TotalCharges,-4683.52,8868.67,0


Decisión aplicada: se conservan todos los registros (0 valores atípicos).


## 3. Revisión de calidad de datos

En la Clase 1 se vieron las dimensiones de la calidad del dato. Revisamos cuatro:

- **Completitud:** que no falten datos.
- **Validez:** que cada valor esté dentro de lo permitido (por ejemplo, `Churn` solo puede ser `Yes` o `No`).
- **Unicidad:** que no haya registros repetidos por error.
- **Consistencia:** que las variables no se contradigan entre sí (por ejemplo, un cliente sin servicio de internet no puede tener soporte técnico contratado).

In [3]:
servicios_internet = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
]
binarias = [
    'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'PaperlessBilling', 'Churn',
]

controles = []


def registrar(dimension, control, cumple):
    controles.append({'dimension': dimension, 'control': control, 'resultado': 'OK' if cumple else 'FALLA'})


# Completitud
registrar('Completitud', 'No hay valores nulos en ninguna columna', not df.isna().any().any())
registrar('Completitud', 'Se mantienen 7.043 clientes y 22 columnas', df.shape == (7043, 22))

# Validez
registrar('Validez', 'Variables binarias solo toman Yes o No', all(df[c].isin(['Yes', 'No']).all() for c in binarias))
registrar('Validez', 'gender solo toma Female o Male', df['gender'].isin(['Female', 'Male']).all())
registrar('Validez', 'Contract tiene las 3 categorías esperadas', set(df['Contract'].unique()) == {'Month-to-month', 'One year', 'Two year'})
registrar('Validez', 'tenure está entre 0 y 72 meses', df['tenure'].between(0, 72).all())
registrar('Validez', 'MonthlyCharges es positivo', (df['MonthlyCharges'] > 0).all())
registrar('Validez', 'TotalCharges no es negativo', (df['TotalCharges'] >= 0).all())

# Consistencia
registrar('Consistencia', 'Sin servicio telefónico, MultipleLines es "No phone service"',
          (df.loc[df['PhoneService'] == 'No', 'MultipleLines'] == 'No phone service').all())
registrar('Consistencia', 'Con servicio telefónico, MultipleLines no es "No phone service"',
          (df.loc[df['PhoneService'] == 'Yes', 'MultipleLines'] != 'No phone service').all())
registrar('Consistencia', 'Sin internet, los 6 servicios de internet son "No internet service"',
          (df.loc[df['InternetService'] == 'No', servicios_internet] == 'No internet service').all().all())
registrar('Consistencia', 'Con internet, ningún servicio es "No internet service"',
          (df.loc[df['InternetService'] != 'No', servicios_internet] != 'No internet service').all().all())
registrar('Consistencia', 'Clientes con tenure = 0 tienen TotalCharges = 0',
          (df.loc[df['tenure'] == 0, 'TotalCharges'] == 0).all())
registrar('Consistencia', 'Clientes con tenure > 0 tienen TotalCharges > 0',
          (df.loc[df['tenure'] > 0, 'TotalCharges'] > 0).all())
registrar('Consistencia', 'ingreso_estimado = MonthlyCharges × tenure',
          np.allclose(df['ingreso_estimado'], df['MonthlyCharges'] * df['tenure']))
registrar('Consistencia', 'diferencia_facturacion = TotalCharges − ingreso_estimado',
          np.allclose(df['diferencia_facturacion'], df['TotalCharges'] - df['ingreso_estimado']))

tabla_controles = pd.DataFrame(controles)
display(tabla_controles)
assert (tabla_controles['resultado'] == 'OK').all(), 'Hay controles de calidad que fallan.'
print(f'{len(tabla_controles)} controles de calidad superados.')

,dimension,control,resultado
0,Completitud,No hay valores nulos en ninguna columna,OK
1,Completitud,Se mantienen 7.043 clientes y 22 columnas,OK
2,Validez,Variables binarias solo toman Yes o No,OK
3,Validez,gender solo toma Female o Male,OK
4,Validez,Contract tiene las 3 categorías esperadas,OK
5,Validez,tenure está entre 0 y 72 meses,OK
6,Validez,MonthlyCharges es positivo,OK
7,Validez,TotalCharges no es negativo,OK
8,Consistencia,"Sin servicio telefónico, MultipleLines es ""No ...",OK
9,Consistencia,"Con servicio telefónico, MultipleLines no es ""...",OK


16 controles de calidad superados.


### Unicidad: filas repetidas

Una fila repetida podría ser un cliente cargado dos veces. Revisamos cuántas hay.

In [4]:
n_duplicadas = int(df.duplicated().sum())
print(f'Filas idénticas (sin contar la primera aparición): {n_duplicadas}')
display(df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(6))

Filas idénticas (sin contar la primera aparición): 22


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,ingreso_estimado,diferencia_facturacion
6491,Female,No,No,No,1,Yes,No,Fiber optic,No,No,...,No,No,Month-to-month,Yes,Electronic check,69.20,69.20,Yes,69.20,0.0
6764,Female,No,No,No,1,Yes,No,Fiber optic,No,No,...,No,No,Month-to-month,Yes,Electronic check,69.20,69.20,Yes,69.20,0.0
4495,Female,No,No,No,1,Yes,No,Fiber optic,No,No,...,No,No,Month-to-month,Yes,Electronic check,70.10,70.10,Yes,70.10,0.0
6267,Female,No,No,No,1,Yes,No,Fiber optic,No,No,...,No,No,Month-to-month,Yes,Electronic check,70.10,70.10,Yes,70.10,0.0
5522,Female,No,No,No,1,Yes,No,Fiber optic,No,No,...,No,No,Month-to-month,Yes,Mailed check,70.15,70.15,Yes,70.15,0.0
5759,Female,No,No,No,1,Yes,No,Fiber optic,No,No,...,No,No,Month-to-month,Yes,Mailed check,70.15,70.15,Yes,70.15,0.0


Hay filas idénticas porque eliminamos `customerID` en la tarea 7. En el dataset original los 7.043 identificadores eran distintos, es decir, **cada fila es un cliente diferente** que casualmente tiene el mismo perfil que otro (mismos servicios, misma antigüedad y mismos cargos). No son errores de carga. **Decisión: conservarlas**, porque eliminarlas haría perder clientes reales y alteraría las proporciones de `Churn`.

## 4. Exportación del dataset limpio

Guardamos `data/processed/telco_clean.csv` sin índice y comprobamos que, al volver a leerlo con el esquema, se recupera exactamente el mismo DataFrame.

In [5]:
RUTA_SALIDA = RAIZ / 'data/processed/telco_clean.csv'
RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(RUTA_SALIDA, index=False, encoding='utf-8')

df_recargado = pd.read_csv(RUTA_SALIDA, dtype=tipos_esperados)
pd.testing.assert_frame_equal(df, df_recargado, check_exact=False, rtol=1e-12, atol=1e-9)

print(f'Archivo generado: {RUTA_SALIDA.relative_to(RAIZ).as_posix()}')
print(f'Salida: {len(df_recargado):,} clientes y {df_recargado.shape[1]} columnas, sin nulos.')
print('Verificación de lectura: valores y tipos recuperados correctamente.')

Archivo generado: data/processed/telco_clean.csv
Salida: 7,043 clientes y 22 columnas, sin nulos.
Verificación de lectura: valores y tipos recuperados correctamente.


**Resultado de la tarea:** el dataset limpio final tiene **7.043 clientes y 22 columnas** (17 categóricas y 5 numéricas), sin valores nulos y sin valores atípicos. Se conservaron todos los registros, incluidas las 22 filas de perfil repetido, que corresponden a clientes distintos. Los 16 controles de completitud, validez y consistencia se cumplen. El archivo `data/processed/telco_clean.csv` queda como entrada de las tareas 10 en adelante (EDA, visualización, preprocesamiento y modelado).

**Recordatorio de uso:** al leerlo, indicar el esquema con `pd.read_csv('data/processed/telco_clean.csv', dtype=tipos_esperados)` para recuperar los tipos `category`.